# Eksplorasi Data (EDA) & Segmentasi Pelanggan (RFM) - Nabila Catering
**Program Mahasiswa Berdampak - Studi Independen PT Vinix Seven Aurum (Vinix7)**
Mahasiswa: S1 Sains Data UNESA

Notebook ini mencakup analisis statistik deskriptif, evaluasi tren omzet bulanan, analisis popularitas menu (menu engineering), dan pemodelan segmentasi pelanggan menggunakan algoritma Recency, Frequency, Monetary (RFM).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Memuat data transaksi Nabila Catering
df = pd.read_csv('../data/historis_transaksi_catering.csv')
df['Tanggal_Pesanan'] = pd.to_datetime(df['Tanggal_Pesanan'])
df['Tanggal_Selesai'] = pd.to_datetime(df['Tanggal_Selesai'])
print('Total Transaksi:', len(df))
df.head()

## 1. Analisis Metrik Finansial Utama
Menghitung Total Omzet, Total Porsi Terjual, dan Average Order Value (AOV).

In [ ]:
total_omzet = df['Total_Penjualan'].sum()
total_porsi = df['Jumlah_Pesanan'].sum()
total_order = len(df)
aov = total_omzet / total_order

print(f'Total Omzet       : Rp {total_omzet:,.0f}')
print(f'Total Porsi       : {total_porsi:,} porsi')
print(f'Rata-rata Order   : Rp {aov:,.0f} per transaksi')

## 2. Analisis Menu Terlaris (Menu Engineering)
Mengidentifikasi menu pendorong volume (volume drivers) dan menu penghasil omzet tertinggi (revenue drivers).

In [ ]:
top_volume = df.groupby('Nama_Menu')['Jumlah_Pesanan'].sum().sort_values(ascending=False).head(10)
top_revenue = df.groupby('Nama_Menu')['Total_Penjualan'].sum().sort_values(ascending=False).head(10)

print('--- Top 5 Menu Volume Terbesar ---')
print(top_volume.head(5))
print('\n--- Top 5 Menu Omzet Terbesar ---')
print(top_revenue.head(5))

## 3. Pemodelan Segmentasi Pelanggan (RFM)
Menghitung skor Recency (hari sejak order terakhir), Frequency (berapa kali repeat order), dan Monetary (total belanja).

In [ ]:
snapshot_date = df['Tanggal_Pesanan'].max() + timedelta(days=1)
rfm = df.groupby('Nama_Pemesan').agg(
    Recency=('Tanggal_Pesanan', lambda x: (snapshot_date - x.max()).days),
    Frequency=('ID_Pesanan', 'count'),
    Monetary=('Total_Penjualan', 'sum')
).reset_index()

def segmentasi(row):
    if row['Frequency'] >= 8:
        return 'Mitra Institusi (B2B Rutin)'
    elif row['Frequency'] >= 3:
        return 'Pelanggan Rutin'
    elif row['Monetary'] >= 3000000:
        return 'Klien Acara Besar'
    elif row['Monetary'] >= 1000000:
        return 'Pemesan Menengah'
    else:
        return 'Pelanggan Retail'

rfm['Segmen'] = rfm.apply(segmentasi, axis=1)
print(rfm['Segmen'].value_counts())
rfm.head()